In [2]:
! pip install -q chromadb sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
! pip install -qU google-genai


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions
# Import the official Gemini SDK
from google import genai 

# 1. Initialize the Gemini Client for text generation / chat
# Ensure your API key is set in your environment: export GEMINI_API_KEY="your-key"
client = genai.Client()

# ==========================================
# STAGE 1: DOCUMENT LOADING
# ==========================================
# Simulating raw document text loaded from an internal source or file
raw_document = (
    "ACME Corp Remote Work Policy. Effective Date: January 2026. "
    "All employees are permitted to work remotely up to 3 days per week. "
    "Core operating hours where everyone must be online are 10:00 AM to 3:00 PM EST. "
    "ACME Corp Expense Reimbursement. "
    "Remote workers can claim a monthly internet allowance of up to $50. "
    "All expense receipts must be submitted via the internal Concur portal by the 25th of each month."
)
print("📦 Stage 1: Document Loaded successfully.")

# ==========================================
# STAGE 2: DOCUMENT CHUNKING
# ==========================================
def chunk_text(text, chunk_size=30, overlap=10):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

# Split our text into small, context-retaining pieces
chunks = chunk_text(raw_document, chunk_size=20, overlap=5)
print(f"🧩 Stage 2: Split text into {len(chunks)} overlapping chunks.")

# ==========================================
# STAGE 3 & 4: EMBEDDING & VECTOR DB INDEXING
# ==========================================
# 1. Initialize an in-memory open-source ChromaDB instance
chroma_client = chromadb.EphemeralClient()
from chromadb.utils import embedding_functions

# 2. Tell ChromaDB to automatically use an open-source embedding model
# This automatically handles converting strings to math vectors under the hood
embedding_model = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Define collection name
collection_name = "company_policies"

# Check if collection exists and delete it for a clean run
try:
    chroma_client.delete_collection(name=collection_name)
    print(f"Deleted existing collection: {collection_name}")
except Exception as e:
    print(f"Collection {collection_name} did not exist or could not be deleted: {e}")

# 3. Create a Vector DB Index (Collection)
collection = chroma_client.create_collection(name=collection_name, embedding_function=embedding_model)

# 4. Insert chunks into the database index
# We generate unique IDs for each chunk to allow indexing
chunk_ids = [f"id_{i}" for i in range(len(chunks))]
collection.add(documents=chunks, ids=chunk_ids)
print("🗂️ Stage 3 & 4: Embedded text chunks and indexed into ChromaDB.")

# ==========================================
# STAGE 5: QUERY SEARCH (RETRIEVAL)
# ==========================================
user_query = "How much can I get back for my home wi-fi bill?"

# Chroma automatically embeds the query using the model defined above,
# runs a similarity search against the index, and grabs the single best match
search_results = collection.query(query_texts=[user_query], n_results=1)

# Extract the matching text piece from Chroma's data structure
retrieved_context = search_results['documents'][0][0]
print(f"🔍 Stage 5: Retrieved nearest match -> \"{retrieved_context}\"")

# ==========================================
# STAGE 6: AUGMENTATION (PROMPT ASSEMBLY)
# ==========================================
system_prompt = (
    "You are an internal HR assistant. Answer the user's question accurately using ONLY "
    "the provided Context. If the context doesn't contain the answer, say you don't know."
)

user_prompt = f"Context:\n{retrieved_context}\n\nQuestion: {user_query}"
print("📝 Stage 6: Augmented the prompt with the retrieved database truth.")

# ==========================================
# STAGE 7: GENERATION (LIVE OPENAI LLM CALL)
# ==========================================
print("🤖 Stage 7: Sending package to Gemini (gemini-2.5-flash)...")
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents=user_prompt,
    config={
        "system_instruction": system_prompt,
        "temperature": 0.0,
    },
)

final_answer = response.text # Fix: Access the first choice in the list
print("\n================ FINAL SYSTEM RESULT ================")
print(f"Query : {user_query}")
print(f"Answer: {response.text}")
print("=====================================================")



📦 Stage 1: Document Loaded successfully.
🧩 Stage 2: Split text into 5 overlapping chunks.
Deleted existing collection: company_policies
🗂️ Stage 3 & 4: Embedded text chunks and indexed into ChromaDB.
🔍 Stage 5: Retrieved nearest match -> "a monthly internet allowance of up to $50. All expense receipts must be submitted via the internal Concur portal by"
📝 Stage 6: Augmented the prompt with the retrieved database truth.
🤖 Stage 7: Sending package to Gemini (gemini-2.5-flash)...

================ FINAL SYSTEM RESULT ================
Query : How much can I get back for my home wi-fi bill?
Answer: Based on the provided context, you can receive a monthly internet allowance of up to $50.


In [1]:
! pip install -q chromadb pypdf


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:

import chromadb
from pathlib import Path
from pypdf import PdfReader
from chromadb.utils import embedding_functions
from google import genai

# Gemini
client = genai.Client()

# 1. LOAD DOCUMENTS
docs = []
for file in Path("D:/AI/docs_dir").iterdir():
    if file.suffix.lower() == ".pdf":
        for i, page in enumerate(PdfReader(file).pages, 1):
            text = page.extract_text() or ""
            if text.strip():
                docs.append((text, file.name, i))
    elif file.suffix.lower() == ".txt":
        docs.append((file.read_text(), file.name, 0))

# 2. CHUNK DOCUMENTS
chunks, metadata = [], []
for text, name, page in docs:
    words = text.split()
    for i in range(0, len(words), 200):
        chunk = " ".join(words[i:i+200])
        if chunk:
            chunks.append(chunk)
            metadata.append({"source": name, "page": page})

# 3. EMBED AND STORE
# 3. EMBED AND STORE
db = chromadb.Client()

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Remove existing collection
try:
    db.delete_collection("docs")
except Exception:
    pass

# Create fresh collection
col = db.create_collection(
    name="docs",
    embedding_function=ef
)

col.add(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks,
    metadatas=metadata
)

# 4. RETRIEVE RELEVANT CHUNKS
question = input("Ask a question: ")
result = col.query(
    query_texts=[question],
    n_results=min(5, len(chunks))
)
context = "\n".join(result["documents"][0])

# 5. GENERATE ANSWER
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=f"Answer using only this context:\n{context}\n\nQuestion: {question}",
    config={"temperature": 0.0}
)
print(response.text)

Based on the provided context, the document states the following about Ecosystem and Evolution:

**Ecosystem:**
* An ecosystem is a functional unit of nature where living organisms interact among themselves and with the surrounding physical environment.
* Ecosystems vary greatly in size, ranging from a small pond to a large forest or sea, and many ecologists regard the entire biosphere as a global ecosystem (a composite of all local ecosystems on Earth).
* Ecosystems are divided into two basic categories: terrestrial (e.g., forest, grassland, desert) and aquatic (e.g., pond, lake, wetland, river, estuary), along with man-made ecosystems like crop fields and aquariums.
* The interaction of biotic and abiotic components results in a physical structure characteristic of each ecosystem type. 
* Identification and enumeration of plant and animal species give its species composition, while the vertical distribution of different species occupying different levels is called stratification.
* T